# 05 · Graph Algorithms — The Honest Limit

AGE is a **pattern-matching and pathfinding engine, not a graph-analytics library**. It
has no built-in PageRank, community detection, or centrality. Pretending otherwise is the
most common way people conclude AGE is "weak" — they are using a traversal engine for an
analytics job.

The honest, productive workflow:

1. **Extract** the relevant subgraph with one Cypher query,
2. **compute** the algorithm in NetworkX / igraph in a few lines,
3. **write the scores back** as node properties.

We do exactly that for PageRank, end to end.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import networkx as nx

from src.age.connect import age_available, get_connection
from src.age.cypher import run_cypher
from src.age.to_networkx import subgraph_to_networkx
from src.config import load_settings
from src.datasets.graphs import KNOWS, all_graphs
from src.outputs import figure_path, save_json
from src.visualisation import draw_graph

settings = load_settings()
AGE = age_available(settings)
social = all_graphs(settings)["social_txn"]
conn = get_connection(settings) if AGE else None
GRAPH = settings.graphs.social
print("AGE:", AGE)

AGE: True


## AGE has no built-in PageRank

Not a criticism — a design choice. We confirm it by looking for any `pagerank` function in
AGE's `ag_catalog` schema: there are none.

In [2]:
if AGE:
    row = conn.execute(
        "SELECT count(*) FROM pg_proc p "
        "JOIN pg_namespace n ON n.oid = p.pronamespace "
        "WHERE n.nspname = 'ag_catalog' AND p.proname ILIKE 'pagerank'"
    ).fetchone()
    print("AGE built-in pagerank functions in ag_catalog:", row[0])
else:
    print("AGE not available; the extract -> compute -> write-back pattern still runs on baselines.")

AGE built-in pagerank functions in ag_catalog: 0


## Step 1 — extract the subgraph with Cypher

One query pulls the `KNOWS` graph out of AGE and hands it to NetworkX (`rows_to_networkx`
is pure, so this bridge is unit-tested).

In [3]:
if AGE:
    extracted = subgraph_to_networkx(conn, GRAPH, "MATCH (a:Account)-[r:KNOWS]->(b:Account)")
    # AGE vertex ids differ from our node_id property; key the graph by node_id for write-back.
    relabel = {n: data["node_id"] for n, data in extracted.nodes(data=True)}
    extracted = nx.relabel_nodes(extracted, relabel)
else:
    extracted = social.to_networkx(KNOWS)

print(f"extracted {extracted.number_of_nodes()} nodes, {extracted.number_of_edges()} edges")

extracted 400 nodes, 3018 edges


## Step 2 — compute PageRank in NetworkX

In [4]:
scores = nx.pagerank(extracted)
top10 = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("top 10 accounts by PageRank:")
for node_id, score in top10:
    print(f"  account {node_id:>3}: {score:.5f}")
save_json("05_pagerank.json", {
    "method": "networkx.pagerank",
    "n_nodes": len(scores),
    "top10": [[int(n), round(float(s), 6)] for n, s in top10],
})

top 10 accounts by PageRank:
  account 306: 0.00429
  account 202: 0.00427
  account 191: 0.00407
  account 223: 0.00404
  account 110: 0.00402
  account  74: 0.00398
  account 281: 0.00397
  account   7: 0.00375
  account 161: 0.00374
  account 119: 0.00370


PosixPath('/workspaces/apache_age_101/outputs/data/05_pagerank.json')

## Step 3 — write the scores back into AGE as node properties

A `SET` per account. Then we read the top accounts back *out of AGE* and confirm they match
what NetworkX computed — the round trip that makes the division of labour real.

In [5]:
if AGE:
    for node_id, score in scores.items():
        run_cypher(conn, GRAPH,
                   f"MATCH (a:Account {{node_id: {node_id}}}) SET a.pagerank = {score}")
    rows = run_cypher(
        conn, GRAPH,
        "MATCH (a:Account) WHERE a.pagerank IS NOT NULL "
        "RETURN a.node_id, a.pagerank ORDER BY a.pagerank DESC LIMIT 10",
        columns=("node_id", "pagerank"),
    )
    age_top10 = [[int(r[0]), round(float(r[1]), 6)] for r in rows]
    nx_top_ids = [int(n) for n, _ in top10]
    age_top_ids = [n for n, _ in age_top10]
    match = set(nx_top_ids) == set(age_top_ids)
    print("AGE read-back top 10 ids:", age_top_ids)
    print("matches NetworkX:", match)
    assert match, "write-back / read-back disagrees with NetworkX"
else:
    age_top10, match = None, None

save_json("05_writeback.json", {
    "networkx_top10": [[int(n), round(float(s), 6)] for n, s in top10],
    "age_readback_top10": age_top10,
    "match": match,
})

AGE read-back top 10 ids: [306, 202, 191, 223, 110, 74, 281, 7, 161, 119]
matches NetworkX: True


PosixPath('/workspaces/apache_age_101/outputs/data/05_writeback.json')

## Figure: the influential neighbourhood

In [6]:
top_node = top10[0][0]
ego_ids = set(nx.ego_graph(extracted.to_undirected(), top_node, radius=1).nodes())
ego = extracted.subgraph(ego_ids)
p = draw_graph(
    ego,
    figure_path("05_pagerank_graph.png"),
    title=f"KNOWS neighbourhood of the top-PageRank account ({top_node})",
    highlight={top_node},
)
print("saved ->", p)

saved -> /workspaces/apache_age_101/outputs/figures/05_pagerank_graph.png


## Figure: PageRank leaderboard

In [7]:
import matplotlib.pyplot as plt

ids = [str(n) for n, _ in top10]
vals = [s for _, s in top10]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ids, vals, color="#4c6ef5")
ax.set_xlabel("account")
ax.set_ylabel("PageRank")
ax.set_title("Top 10 accounts by PageRank (computed in NetworkX, stored in AGE)")
fig.tight_layout()
fig.savefig(figure_path("05_top_pagerank.png"), dpi=120)
plt.close(fig)
if conn is not None:
    conn.close()
print("saved leaderboard")

saved leaderboard


## And one more thing: pgvector lives in the same Postgres

Because AGE is just a Postgres extension, **pgvector** can sit in the same database. That
means graph traversal (AGE) and vector search (pgvector) over one store — hybrid
graph-plus-vector retrieval with no second system. This is the bridge to the
`graphrag_vs_vectorrag` and `vector_search_101` projects.

**Takeaway.** AGE traverses and matches; Python analyses. Extract a subgraph with Cypher,
run PageRank (or communities, or centrality) in NetworkX/igraph, write the scores back.
Knowing this division of labour is what stops you fighting the wrong tool.